# One-Generation Tool Selection Inspection

Run one selected test example through OpenAI, Gemini, Claude, local Qwen, and NTILC with a 50-tool candidate catalog containing the correct tool plus 49 seeded random distractors. The model cells make live API/local model calls when the required keys, dependencies, checkpoints, and hardware are available.


In [1]:
from __future__ import annotations

import os
import random
import sys
import time
from pathlib import Path
from typing import Any, Callable

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    display = print

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'benchmark').is_dir():
    for parent in REPO_ROOT.parents:
        if (parent / 'benchmark').is_dir():
            REPO_ROOT = parent
            break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from benchmark.adapters import (
    AnthropicSelectionAdapter,
    EmbeddingSelectionAdapter,
    GeminiSelectionAdapter,
    LocalHFSelectionAdapter,
    OpenAISelectionAdapter,
)
from benchmark.common import DEFAULT_RANKING_LIMIT, build_selection_messages, load_benchmark_rows, load_tool_catalog
from benchmark.env import load_env_file
from benchmark.main_inference_time_comparison import (
    DEFAULT_ANTHROPIC_MODEL,
    DEFAULT_GEMINI_MODEL,
    DEFAULT_NTILC_CHECKPOINT_GLOB,
    DEFAULT_OPENAI_MODEL,
    DEFAULT_QWEN_MODEL,
    infer_embedding_variant,
)

DATASET_NAME = 'MetaTool'
ROW_INDEX = 0
CANDIDATE_TOOL_COUNT = 50
SEED = 7
RANKING_LIMIT = DEFAULT_RANKING_LIMIT
API_MAX_OUTPUT_TOKENS = None
API_TIMEOUT_SECONDS = 60
DOTENV_PATH = REPO_ROOT / '.env'

NOTEBOOK_DEFAULT_OPENAI_MODEL = DEFAULT_OPENAI_MODEL
NOTEBOOK_DEFAULT_GEMINI_MODEL = 'gemini-2.5-flash'
NOTEBOOK_DEFAULT_ANTHROPIC_MODEL = 'claude-sonnet-4-6'
NOTEBOOK_DEFAULT_QWEN_MODEL = DEFAULT_QWEN_MODEL

loaded_env_keys = load_env_file(DOTENV_PATH)
OPENAI_MODEL = os.getenv('OPENAI_MODEL', os.getenv('OPENAI_MODEL_NAME', NOTEBOOK_DEFAULT_OPENAI_MODEL)).strip()
GEMINI_MODEL = os.getenv('GEMINI_MODEL', os.getenv('GEMINI_MODEL_NAME', NOTEBOOK_DEFAULT_GEMINI_MODEL)).strip()
ANTHROPIC_MODEL = os.getenv('ANTHROPIC_MODEL', os.getenv('ANTHROPIC_MODEL_NAME', NOTEBOOK_DEFAULT_ANTHROPIC_MODEL)).strip()
QWEN_MODEL = os.getenv('QWEN_MODEL', os.getenv('QWEN_MODEL_NAME', NOTEBOOK_DEFAULT_QWEN_MODEL)).strip()
QWEN_DEVICE = os.getenv('QWEN_DEVICE', os.getenv('HF_DEVICE', 'cuda:7')).strip()
QWEN_DTYPE = os.getenv('QWEN_DTYPE', os.getenv('HF_DTYPE', 'auto')).strip()
QWEN_MAX_NEW_TOKENS = int(os.getenv('QWEN_MAX_NEW_TOKENS', os.getenv('HF_MAX_NEW_TOKENS', '160')))
QWEN_LOCAL_FILES_ONLY = os.getenv('QWEN_LOCAL_FILES_ONLY', os.getenv('HF_LOCAL_FILES_ONLY', '0')).strip().lower() in {'1', 'true', 'yes', 'on'}
NTILC_CHECKPOINT_GLOB = os.getenv('NTILC_CHECKPOINT_GLOB', DEFAULT_NTILC_CHECKPOINT_GLOB).strip()
EMBEDDING_DEVICE = os.getenv('EMBEDDING_DEVICE', 'cuda:7').strip()

print(f'Repo root: {REPO_ROOT}')
print(f'Loaded {len(loaded_env_keys)} environment keys from .env; secret values are not printed.')
print('Active model IDs:')
print(f'  OpenAI:    {OPENAI_MODEL}  (main default: {DEFAULT_OPENAI_MODEL}; override with OPENAI_MODEL)')
print(f'  Gemini:    {GEMINI_MODEL}  (main default: {DEFAULT_GEMINI_MODEL}; override with GEMINI_MODEL)')
print(f'  Anthropic: {ANTHROPIC_MODEL}  (main default: {DEFAULT_ANTHROPIC_MODEL}; override with ANTHROPIC_MODEL)')
print(f'  Qwen:      {QWEN_MODEL}  (main default: {DEFAULT_QWEN_MODEL}; override with QWEN_MODEL)')
print(f'  NTILC:     {NTILC_CHECKPOINT_GLOB}  (override with NTILC_CHECKPOINT_GLOB)')


/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: /scratch4/home/akrik/NTILC
Loaded 0 environment keys from .env; secret values are not printed.
Active model IDs:
  OpenAI:    gpt-5.2-2025-12-11  (main default: gpt-5.2-2025-12-11; override with OPENAI_MODEL)
  Gemini:    gemini-2.5-flash  (main default: gemini-2.5-flash; override with GEMINI_MODEL)
  Anthropic: claude-sonnet-4-6  (main default: claude-sonnet-4-6; override with ANTHROPIC_MODEL)
  Qwen:      Qwen/Qwen3.5-27B  (main default: Qwen/Qwen3.5-27B; override with QWEN_MODEL)
  NTILC:     output/normal/functional_margin/**/best.pt  (override with NTILC_CHECKPOINT_GLOB)


In [7]:
def table(rows: list[dict[str, Any]]):
    if pd is not None:
        return pd.DataFrame(rows)
    return rows


def tool_name(tool: dict[str, Any]) -> str:
    return str(tool.get('name', '')).strip()


def build_candidate_tools(
    *,
    dataset_name: str,
    row_index: int,
    benchmark_rows: list[dict[str, Any]],
    tools: list[dict[str, Any]],
    candidate_tool_count: int,
    seed: int,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    if row_index < 0 or row_index >= len(benchmark_rows):
        raise IndexError(f'ROW_INDEX must be between 0 and {len(benchmark_rows) - 1}.')
    if candidate_tool_count < 1:
        raise ValueError('CANDIDATE_TOOL_COUNT must be at least 1.')

    selected_row = dict(benchmark_rows[row_index])
    expected_tool = str(selected_row.get('tool', '')).strip()
    tool_by_name = {tool_name(tool): tool for tool in tools if tool_name(tool)}
    if expected_tool not in tool_by_name:
        raise ValueError(f'Expected tool {expected_tool!r} was not found in tools.json.')

    distractor_pool = [tool for tool in tools if tool_name(tool) and tool_name(tool) != expected_tool]
    distractor_count = candidate_tool_count - 1
    if len(distractor_pool) < distractor_count:
        raise ValueError(
            f'Need {distractor_count} distractors, but only {len(distractor_pool)} are available.'
        )

    rng = random.Random(f'{dataset_name}:{row_index}:{seed}')
    candidates = [tool_by_name[expected_tool], *rng.sample(distractor_pool, distractor_count)]
    rng.shuffle(candidates)
    return selected_row, candidates


def provider_error_hint(provider: str, model_name: str, error_message: str | None) -> str:
    message = str(error_message or '')
    if 'insufficient_quota' in message:
        if provider == 'openai':
            return 'OpenAI key is valid, but the project has no usable quota/billing for generation. Changing models will not fix this key.'
        return 'Provider key is valid, but this project/model has no usable generation quota. Use another key/project/model tier with quota.'
    if 'free_tier_requests' in message or 'free_tier_input_token_count' in message:
        return f'{provider} free-tier quota is exhausted or zero for {model_name!r}. Try a lower-tier model or a paid project/key.'
    if 'HTTP 429' in message:
        return 'Rate limit/quota from the provider. Wait and retry, reduce calls, or use a model/project with available quota.'
    if 'HTTP 404' in message:
        if provider == 'gemini':
            return f'Gemini model or endpoint was not found. Active model is {model_name!r}; set GEMINI_MODEL in .env to a model available to this API key.'
        if provider == 'anthropic':
            return f'Anthropic model or endpoint was not found. Active model is {model_name!r}; set ANTHROPIC_MODEL in .env to a model available to this API key.'
        if provider == 'openai':
            return f'OpenAI model or base URL was not found. Active model is {model_name!r}; set OPENAI_MODEL or OPENAI_BASE_URL in .env if needed.'
        return f'Model or endpoint was not found for {provider}. Check the active model ID and base URL.'
    if 'torch is required' in message or 'transformers is required' in message or 'outlines is required' in message:
        return 'Local Qwen needs the local ML dependencies installed in this notebook kernel.'
    if 'checkpoint' in message.lower() or 'best.pt' in message:
        return 'NTILC needs a dataset checkpoint matching NTILC_CHECKPOINT_GLOB.'
    return ''


def make_error_result(provider: str, model_name: str, row: dict[str, Any], exc: Exception, latency_ms: float | None = None) -> dict[str, Any]:
    error_message = str(exc)
    return {
        'adapter_id': provider,
        'provider': provider,
        'mode': 'unknown',
        'model_name': model_name,
        'example_id': str(row.get('id', '')),
        'query': str(row.get('query', '')),
        'expected_tool': str(row.get('tool', '')).strip(),
        'status': 'error',
        'selected_tool': None,
        'ranked_tools': [],
        'correct_top1': None,
        'latency_ms': latency_ms,
        'input_tokens': None,
        'output_tokens': None,
        'total_tokens': None,
        'reason': '',
        'raw_response': None,
        'score_candidates': None,
        'error_message': error_message,
        'diagnostic': provider_error_hint(provider, model_name, error_message),
    }


def evaluate_selection_model(
    *,
    provider: str,
    model_name: str,
    adapter_factory: Callable[[], Any],
    row: dict[str, Any],
    candidate_tools: list[dict[str, Any]],
) -> dict[str, Any]:
    start_time = time.perf_counter()
    try:
        adapter = adapter_factory()
        summary, results = adapter.evaluate([row], candidate_tools)
        result = dict(results[0]) if results else make_error_result(
            provider,
            model_name,
            row,
            RuntimeError('Adapter returned no result rows.'),
        )
        result['summary_status'] = summary.get('status')
        result['summary_error_message'] = summary.get('error_message')
        error_message = result.get('error_message') or result.get('summary_error_message')
        result['diagnostic'] = provider_error_hint(provider, model_name, error_message)
        return result
    except Exception as exc:
        latency_ms = round((time.perf_counter() - start_time) * 1000.0, 6)
        return make_error_result(provider, model_name, row, exc, latency_ms=latency_ms)


def resolve_ntilc_variant(dataset_dir: Path):
    checkpoint_matches = sorted(dataset_dir.glob(NTILC_CHECKPOINT_GLOB))
    if not checkpoint_matches:
        raise FileNotFoundError(
            f'NTILC checkpoint not found. Looked for {NTILC_CHECKPOINT_GLOB!r} under {dataset_dir}.'
        )
    return infer_embedding_variant(dataset_dir, checkpoint_matches[0])


In [3]:
dataset_dir = REPO_ROOT / 'data' / DATASET_NAME
dataset_path = dataset_dir / 'tool_embedding_dataset_test.jsonl'
tools_path = dataset_dir / 'tools.json'

benchmark_rows = load_benchmark_rows(dataset_path)
tools = load_tool_catalog(tools_path)
selected_row, candidate_tools = build_candidate_tools(
    dataset_name=DATASET_NAME,
    row_index=ROW_INDEX,
    benchmark_rows=benchmark_rows,
    tools=tools,
    candidate_tool_count=CANDIDATE_TOOL_COUNT,
    seed=SEED,
)
candidate_names = [tool_name(tool) for tool in candidate_tools]
expected_tool = str(selected_row['tool']).strip()
prompt_messages = build_selection_messages(
    str(selected_row['query']),
    candidate_tools,
    ranking_limit=RANKING_LIMIT,
)

try:
    ntilc_variant = resolve_ntilc_variant(dataset_dir)
    ntilc_model_name = ntilc_variant.variant_id
    ntilc_checkpoint_path = str(ntilc_variant.checkpoint_path)
except Exception as exc:
    ntilc_variant = None
    ntilc_model_name = 'ntilc'
    ntilc_checkpoint_path = str(exc)

display(table([
    {
        'dataset': DATASET_NAME,
        'dataset_rows': len(benchmark_rows),
        'catalog_tools': len(tools),
        'row_index': ROW_INDEX,
        'example_id': selected_row.get('id', ''),
        'query': selected_row['query'],
        'expected_tool': expected_tool,
        'candidate_count': len(candidate_tools),
        'expected_position_1_based': candidate_names.index(expected_tool) + 1,
        'ranking_limit': RANKING_LIMIT,
        'prompt_messages': len(prompt_messages),
        'ntilc_checkpoint': ntilc_checkpoint_path,
    }
]))

display(table([
    {
        'position': index,
        'tool': name,
        'is_expected': name == expected_tool,
        'description': str(candidate_tools[index - 1].get('description', '')),
    }
    for index, name in enumerate(candidate_names, start=1)
]))


,dataset,dataset_rows,catalog_tools,row_index,example_id,query,expected_tool,candidate_count,expected_position_1_based,ranking_limit,prompt_messages,ntilc_checkpoint
0,MetaTool,796,199,0,buildbetter-0017,Show me the highlights from my meetings in the...,buildbetter,50,9,5,2,/scratch4/home/akrik/NTILC/data/MetaTool/outpu...


,position,tool,is_expected,description
0,1,dover_outreach,False,Generate a personalized email to someone you'r...
1,2,review,False,Analyze and summarize reviews
2,3,horoscopes_by_inner_self,False,Daily
3,4,house_purchasing_tool,False,Tool that provides all sorts of information ab...
4,5,weather_tool,False,Provide you with the latest weather information.
5,6,scene_xplain,False,SceneXplain lets you attach images to your pro...
6,7,photorealistic,False,Generate Photorealistic prompts for Midjourney.
7,8,my_writing_companion,False,"Find writing resources, templates, or examples..."
8,9,buildbetter,True,Chat with the knowledge of all your calls in B...
9,10,visla,False,Create a short video from public stock footage...


In [8]:
MODEL_SPECS = [
    {
        'provider': 'openai',
        'model_name': OPENAI_MODEL,
        'adapter_factory': lambda: OpenAISelectionAdapter(
            OPENAI_MODEL,
            ranking_limit=RANKING_LIMIT,
            max_output_tokens=API_MAX_OUTPUT_TOKENS,
            timeout_seconds=API_TIMEOUT_SECONDS,
            pricing=None,
        ),
    },
    {
        'provider': 'gemini',
        'model_name': GEMINI_MODEL,
        'adapter_factory': lambda: GeminiSelectionAdapter(
            GEMINI_MODEL,
            ranking_limit=RANKING_LIMIT,
            max_output_tokens=API_MAX_OUTPUT_TOKENS,
            timeout_seconds=API_TIMEOUT_SECONDS,
            pricing=None,
        ),
    },
    {
        'provider': 'anthropic',
        'model_name': ANTHROPIC_MODEL,
        'adapter_factory': lambda: AnthropicSelectionAdapter(
            ANTHROPIC_MODEL,
            ranking_limit=RANKING_LIMIT,
            max_output_tokens=API_MAX_OUTPUT_TOKENS,
            timeout_seconds=API_TIMEOUT_SECONDS,
            pricing=None,
        ),
    },
    {
        'provider': 'qwen',
        'model_name': QWEN_MODEL,
        'adapter_factory': lambda: LocalHFSelectionAdapter(
            QWEN_MODEL,
            device=QWEN_DEVICE,
            dtype=QWEN_DTYPE,
            ranking_limit=RANKING_LIMIT,
            max_new_tokens=QWEN_MAX_NEW_TOKENS,
            local_files_only=QWEN_LOCAL_FILES_ONLY,
            pricing=None,
        ),
    },
    {
        'provider': 'ntilc',
        'model_name': ntilc_model_name,
        'adapter_factory': lambda: EmbeddingSelectionAdapter(
            ntilc_variant,
            device=EMBEDDING_DEVICE,
            ranking_limit=RANKING_LIMIT,
        ) if ntilc_variant is not None else (_ for _ in ()).throw(FileNotFoundError(ntilc_checkpoint_path)),
    },
]

one_generation_results: list[dict[str, Any]] = []
for spec in MODEL_SPECS:
    print(f"Running {spec['provider']} / {spec['model_name']} ...")
    result = evaluate_selection_model(
        provider=spec['provider'],
        model_name=spec['model_name'],
        adapter_factory=spec['adapter_factory'],
        row=selected_row,
        candidate_tools=candidate_tools,
    )
    one_generation_results.append(result)
    display(table([
        {
            'provider': result.get('provider'),
            'mode': result.get('mode'),
            'model_name': result.get('model_name'),
            'status': result.get('status'),
            'error_message': result.get('error_message') or result.get('summary_error_message'),
            'diagnostic': result.get('diagnostic'),
            'selected_tool': result.get('selected_tool'),
            'expected_tool': result.get('expected_tool'),
            'correct_top1': result.get('correct_top1'),
            'ranked_tools': result.get('ranked_tools'),
            'reason': result.get('reason'),
            'input_tokens': result.get('input_tokens'),
            'output_tokens': result.get('output_tokens'),
            'total_tokens': result.get('total_tokens'),
            'latency_ms': result.get('latency_ms'),
        }
    ]))
    print('raw_response:')
    print(result.get('raw_response') or '')
    if result.get('score_candidates'):
        print('score_candidates:')
        print(result.get('score_candidates'))
    print()


Running openai / gpt-5.2-2025-12-11 ...


Benchmarking openai/gpt-5-2-2025-12-11: 100%|██████████| 1/1 [00:01<00:00,  1.84s/example]


,provider,mode,model_name,status,error_message,diagnostic,selected_tool,expected_tool,correct_top1,ranked_tools,reason,input_tokens,output_tokens,total_tokens,latency_ms
0,openai,llm_api,gpt-5.2-2025-12-11,ok,None,,buildbetter,buildbetter,True,"[buildbetter, chat_with_workspace, data_retrie...",buildbetter is designed to query and summarize...,2594,73,2667,1833.037622


raw_response:
{"selected_tool":"buildbetter","ranked_tools":["buildbetter","chat_with_workspace","data_retrieval_tool","exportchat","research_helper"],"reason":"buildbetter is designed to query and summarize Zoom/meeting call data over a specified date range, which fits extracting last quarter meeting highlights."}

Running gemini / gemini-2.5-flash ...


Benchmarking gemini/gemini-2-5-flash: 100%|██████████| 1/1 [00:02<00:00,  2.47s/example]


...

,provider,mode,model_name,status,error_message,diagnostic,selected_tool,expected_tool,correct_top1,ranked_tools,reason,input_tokens,output_tokens,total_tokens,latency_ms
0,huggingface,llm_local,Qwen/Qwen3.5-27B,ok,None,,buildbetter,buildbetter,True,"[buildbetter, data_retrieval_tool, chat_with_w...",The buildbetter tool is specifically designed ...,1846,83,1929,5021.041314


raw_response:
{ "selected_tool": "buildbetter", "ranked_tools": ["buildbetter", "data_retrieval_tool", "chat_with_workspace", "exportchat", "reflect_notes"], "reason": "The buildbetter tool is specifically designed to chat with the knowledge of all your calls in BuildBetter (Zoom), which aligns with the user's request to show highlights from meetings in the last quarter."}

Running ntilc / normal/functional_margin ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1346.56it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Benchmarking normal/functional_margin: 100%|██████████| 1/1 [00:00<00:00, 207.14example/s]


,provider,mode,model_name,status,error_message,diagnostic,selected_tool,expected_tool,correct_top1,ranked_tools,reason,input_tokens,output_tokens,total_tokens,latency_ms
0,embedding,embedding,normal/functional_margin,ok,None,,buildbetter,buildbetter,True,"[buildbetter, chat_with_workspace, magnetis, r...",Embedding nearest-neighbor selection.,14,0,14,3.52044


raw_response:

score_candidates:
[{'tool': 'buildbetter', 'score': 0.600198}, {'tool': 'chat_with_workspace', 'score': 0.426577}, {'tool': 'magnetis', 'score': 0.329871}, {'tool': 'review', 'score': 0.239696}, {'tool': 'shopping_assistant', 'score': 0.235487}]



In [5]:
comparison_rows = [
    {
        'provider': result.get('provider'),
        'mode': result.get('mode'),
        'model_name': result.get('model_name'),
        'status': result.get('status'),
        'selected_tool': result.get('selected_tool'),
        'expected_tool': result.get('expected_tool'),
        'correct_top1': result.get('correct_top1'),
        'input_tokens': result.get('input_tokens'),
        'output_tokens': result.get('output_tokens'),
        'total_tokens': result.get('total_tokens'),
        'latency_ms': result.get('latency_ms'),
        'error_message': result.get('error_message') or result.get('summary_error_message'),
        'diagnostic': result.get('diagnostic'),
    }
    for result in one_generation_results
]

display(table(comparison_rows))


,provider,mode,model_name,status,selected_tool,expected_tool,correct_top1,input_tokens,output_tokens,total_tokens,latency_ms,error_message,diagnostic
0,openai,llm_api,gpt-5.2-2025-12-11,ok,buildbetter,buildbetter,True,2594,71,2665,2840.030724,None,
1,gemini,llm_api,gemini-2.5-flash,ok,buildbetter,buildbetter,True,2032,87,2119,1161.149761,None,
2,anthropic,llm_api,claude-sonnet-4-6,ok,buildbetter,buildbetter,True,3520,133,3653,3182.129529,None,
3,huggingface,llm_local,Qwen/Qwen3.5-27B,ok,buildbetter,buildbetter,True,1846,83,1929,10511.003180,None,
4,embedding,embedding,normal/functional_margin,ok,buildbetter,buildbetter,True,14,0,14,42.375332,None,


In [6]:
comparison_rows[4]

{'provider': 'embedding',
 'mode': 'embedding',
 'model_name': 'normal/functional_margin',
 'status': 'ok',
 'selected_tool': 'buildbetter',
 'expected_tool': 'buildbetter',
 'correct_top1': True,
 'input_tokens': 14,
 'output_tokens': 0,
 'total_tokens': 14,
 'latency_ms': 42.375332,
 'error_message': None,
 'diagnostic': ''}